In [ ]:
# 1. 安裝必要套件
!pip install pandas numpy scikit-learn xgboost shap matplotlib openpyxl

In [ ]:
# 2. 載入資料
import pandas as pd
import numpy as np

import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 3. 準備特徵與目標
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 4. 標準化資料
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 5. Lasso 進行特徵選擇
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)
coef = pd.Series(lasso.coef_, index=X.columns)
selected_features = coef[coef != 0].index.tolist()
print('Lasso 選出的特徵:', selected_features)

# 直接用 Lasso 特徵，沒有就用全部特徵
if len(selected_features) > 0:
    final_features = selected_features
else:
    print("Lasso未選出任何特徵，改為使用所有特徵進行分析")
    final_features = X.columns.tolist()

# 6. SHAP 解釋
import shap
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_scaled[final_features], y)

explainer = shap.Explainer(model, X_scaled[final_features])
shap_values = explainer(X_scaled[final_features])

shap.summary_plot(shap_values, X_scaled[final_features], show=False)
plt.title('SHAP Summary for Lasso-selected Features')
plt.show()

# 7. 分組訓練與測試
from sklearn.model_selection import train_test_split

teams = df['Team'].tolist()
np.random.seed(42)
test_teams = np.random.choice(teams, size=5, replace=False)
train_teams = [t for t in teams if t not in test_teams]

X_train = X_scaled[df['Team'].isin(train_teams)][final_features]
y_train = y[df['Team'].isin(train_teams)]
X_test = X_scaled[df['Team'].isin(test_teams)][final_features]
y_test = y[df['Team'].isin(test_teams)]

# 8. XGBoost 訓練與預測
import xgboost as xgb
from sklearn.metrics import r2_score

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'訓練組 R2: {train_r2:.3f}')
print(f'測試組 R2: {test_r2:.3f}')

result = pd.DataFrame({
    'Team': np.array(test_teams),
    '實際勝率': y_test.values,
    '預測勝率': y_test_pred,
    '誤差': y_test_pred - y_test.values
})
print(result)

# 9. XGBoost SHAP
explainer_xgb = shap.Explainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb(X_test)
shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.title('XGBoost SHAP Summary (Test Set)')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV, lasso_path
from sklearn.preprocessing import StandardScaler

# 範例資料載入與標準化（你可以替換成自己的資料）
# 這裡我們建立假資料模擬
np.random.seed(42)
X = pd.DataFrame(np.random.randn(100, 10), columns=[f'Feature_{i}' for i in range(10)])
y = X['Feature_0'] * 0.5 - X['Feature_3'] * 1.2 + np.random.randn(100) * 0.5  # 模擬有兩個重要特徵

# 資料標準化
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# LassoCV 訓練
alphas = np.logspace(-3, 1, 100)
lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_scaled, y)

# 選出非零係數特徵
coef = pd.Series(lasso_cv.coef_, index=X.columns)
selected_features = coef[coef != 0].index.tolist()

# 繪製 Lasso path
alphas_path, coefs_path, _ = lasso_path(X_scaled, y, alphas=alphas)

plt.figure(figsize=(10, 6))
for i in range(coefs_path.shape[0]):
    plt.plot(alphas_path, coefs_path[i], label=X.columns[i])
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Path')
plt.legend(loc='upper right', fontsize=8)
plt.grid(True)
plt.tight_layout()

# 回傳結果表格
result_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lasso_cv.coef_,
    'Selected': lasso_cv.coef_ != 0
})

import ace_tools as tools; tools.display_dataframe_to_user(name="Lasso 特徵選擇結果", dataframe=result_df)
